# DRAS-5 State Machine Basics

This notebook demonstrates the core features of the DRAS-5 five-state
risk assessment machine, including all five safety constraints (C1--C5)
and the exponential decay de-escalation protocol.

## Outline

1. State parameters (Table 2)
2. Gradual risk escalation
3. Monotonic escalation (C1)
4. Human approval gate (C4)
5. C5 controlled de-escalation
6. Exponential risk decay
7. Audit trail (C3)
8. Trajectory simulation

## Setup

In [ ]:
import sys, math
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dras5 import (
    DRAS5StateMachine, RiskState, STATE_CONFIG,
    half_life, min_deescalation_latency, check_c5,
    generate_trajectory,
)

plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 120})
print("DRAS-5 v1.0.0 imported successfully")

## 1. State Parameters (Table 2)

In [ ]:
# Display Table 2 parameters
rows = []
for s in RiskState:
    cfg = STATE_CONFIG[s]
    t12 = half_life(s)
    rows.append({
        "State": s.name,
        "theta": cfg["theta"],
        "T_max (s)": cfg["t_max"] if cfg["t_max"] != float("inf") else "inf",
        "lambda (s^-1)": cfg["lam"] if cfg["lam"] else "---",
        "T_cool (s)": cfg["t_cool"] if cfg["t_cool"] else "---",
        "t_1/2 (s)": f"{t12:.1f}" if t12 else "---",
    })

df_params = pd.DataFrame(rows)
print("Table 2: DRAS-5 State Parameters\n")
print(df_params.to_string(index=False))
print(f"\nNote: S4 decays ~{half_life(RiskState.CRITICAL)/half_life(RiskState.MONITOR):.0f}x slower than S2")

## 2. Gradual Risk Escalation

In [ ]:
sm = DRAS5StateMachine(require_human_approval=True, session_id="notebook-demo")

scenarios = [
    (0.15,   5, "Normal vitals"),
    (0.35,  15, "Slightly elevated"),
    (0.55,  25, "Moderate concern"),
    (0.75,  35, "Critical signs"),
    (0.95,  45, "Emergency (needs approval)"),
]

print("Gradual escalation demo:\n")
for rho, t, desc in scenarios:
    approved = (rho >= 0.9)
    s = sm.update(risk_score=rho, t=t, human_approved=approved)
    print(f"  rho={rho:.2f}  t={t:3d}s  approved={str(approved):5s}  ->  {s.name:12s}  ({desc})")

## 3. Monotonic Escalation (C1) and Human Gate (C4)

In [ ]:
# C1: Monotonic escalation — no auto-downgrade
sm2 = DRAS5StateMachine(require_human_approval=True)
sm2.update(risk_score=0.75, t=10)
s = sm2.update(risk_score=0.20, t=20)
print(f"C1 test: after rho=0.75 then rho=0.20 -> {s.name} (stayed CRITICAL)")

# C4: Human approval gate — S4->S5 blocked without approval
s = sm2.update(risk_score=0.95, t=30, human_approved=False)
print(f"C4 test: rho=0.95 without approval -> {s.name} (blocked)")

s = sm2.update(risk_score=0.95, t=31, human_approved=True)
print(f"C4 test: rho=0.95 with approval   -> {s.name} (approved)")

## 4. C5 Controlled De-escalation and Exponential Decay

In [ ]:
# C5 de-escalation: ALERT -> MONITOR
sm3 = DRAS5StateMachine(require_human_approval=False)
sm3.update(risk_score=0.55, t=0)  # enter ALERT
print(f"Initial state: {sm3.current_state.name}")

# Need rho_eff < theta_2 = 0.30 over entire cooling period
series_ok = [0.25, 0.22, 0.20, 0.18, 0.15]
ok, msg = check_c5(RiskState.ALERT, series_ok, alpha1=True, alpha2=True)
print(f"C5 check: {msg}")

sm3.update(risk_score=0.15, t=50,
           deescalation_request=True,
           human_approved=True, dual_approval=True,
           rho_eff_series=series_ok)
print(f"After C5: {sm3.current_state.name}")

# Exponential risk decay visualization
print("\nExponential decay from rho_peak=0.85 (S4, lambda=0.001):")
rho_peak = 0.85
lam4 = 0.001
for dt in [0, 100, 200, 400, 693, 800]:
    decayed = rho_peak * math.exp(-lam4 * dt)
    print(f"  t={dt:4d}s  decay={decayed:.4f}  {'(half-life)' if abs(dt - 693) < 1 else ''}")

## 5. Audit Trail (C3)

In [ ]:
# Display audit trail from the first state machine
history = sm.get_history()
df_audit = pd.DataFrame([
    {
        "id": e.entry_id,
        "from": e.from_state.name,
        "to": e.to_state.name,
        "rho": f"{e.risk_score:.3f}",
        "trigger": e.trigger,
        "t": f"{e.timestamp:.1f}",
    }
    for e in history
])

print(f"Audit log: {len(history)} transitions\n")
print(df_audit.to_string(index=False))
print(f"\nJSON export preview:\n{sm.audit_log.to_json()[:300]}...")

## 6. Trajectory Simulation and Visualization

In [ ]:
# Generate and plot three trajectory types
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)

colors = {"SAFE": "#2ecc71", "MONITOR": "#f1c40f", "ALERT": "#e67e22",
          "CRITICAL": "#e74c3c", "EMERGENCY": "#1a1a2e"}

for ax, tt in zip(axes, ["monotonic", "oscillating", "spike_recover"]):
    traj = generate_trajectory(ttype=tt, n_steps=80, dt=10, seed=42)
    t_vals = [p.t for p in traj]
    rho_vals = [p.rho for p in traj]
    state_vals = [p.system_state.name for p in traj]

    # Color-coded scatter by system state
    for sname, color in colors.items():
        mask = [s == sname for s in state_vals]
        t_sub = [t for t, m in zip(t_vals, mask) if m]
        r_sub = [r for r, m in zip(rho_vals, mask) if m]
        if t_sub:
            ax.scatter(t_sub, r_sub, c=color, s=8, label=sname, zorder=3)

    ax.plot(t_vals, rho_vals, color="#aaa", lw=0.5, zorder=2)
    for th, ls in [(0.3, ":"), (0.5, ":"), (0.7, ":"), (0.9, ":")]:
        ax.axhline(th, color="#ddd", ls=ls, lw=0.5)
    ax.set_title(tt.replace("_", "-"), fontsize=10)
    ax.set_xlabel("Time (s)")
    if ax == axes[0]:
        ax.set_ylabel("Risk score")

axes[0].legend(fontsize=6, ncol=2, loc="upper left")
fig.suptitle("Synthetic Trajectory Types (coloured by DRAS-5 state)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

- **Table 2** parameters: thresholds, timeouts, decay rates, cooling periods
- **C1** Monotonic Escalation: no automatic state regression
- **C4** Human Approval Gate: S4->S5 requires clinician approval
- **C5** Controlled De-escalation: sustained decay + dual approval
- **C3** Audit Trail: every transition logged immutably
- Trajectory simulation with three deterioration patterns

**Run the full test suite**: `python -m pytest tests/ -v` (103 tests)

**Generate manuscript figures**: `python scripts/generate_figures.py`